# Dasar Machine Learning

## Bagian 1c: Latihan Exploratory Data Analysis


## Pendahuluan

Latihan ini memakai dataset iris, kumpulan data klasik berisi tiga spesies bunga iris beserta ukuran sepal dan petalnya. Data tersedia pada berkas `iris_data.csv`.


In [ ]:
# Imports
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")


## Pertanyaan 1

Muat data dari berkas, lalu periksa isinya. Tentukan:

- Jumlah baris data (petunjuk: atribut `.shape`).
- Nama kolom (petunjuk: atribut `.columns`).
- Tipe data setiap kolom (petunjuk: atribut `.dtypes`).


In [ ]:
# Unduh berkas sekali saja, lalu muat ke DataFrame
data_path = Path("iris_data.csv")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/iris_data.csv"

if not data_path.exists():
    urllib.request.urlretrieve(data_url, data_path)

dataset = pd.read_csv(data_path)
dataset.head()


In [ ]:
# Jumlah baris, nama kolom, dan tipe data
print(f"Jumlah baris: {dataset.shape[0]}")
print(f"Kolom: {dataset.columns.tolist()}")
print(dataset.dtypes)


Dataset berisi 150 baris dengan lima kolom: empat kolom numerik berisi ukuran bunga dan satu kolom teks berisi nama spesies.


## Pertanyaan 2

Perhatikan nama spesies pada data. Bila masih memuat awalan `Iris-`, potong awalan tersebut agar namanya lebih ringkas. Kamu bisa memakai metode string pandas atau `apply`.


In [ ]:
# Buang awalan 'Iris-' bila ada
dataset["species"] = dataset["species"].str.replace("Iris-", "", regex=False)
dataset["species"].unique()


## Pertanyaan 3

Tentukan:

- Jumlah data untuk setiap spesies (petunjuk: metode `.value_counts`).
- Rata-rata, median, kuartil, serta rentang (maksimum dikurangi minimum) setiap ukuran petal dan sepal.

Petunjuk: `.describe` sudah memuat median dalam bentuk kuantil *50%*, tetapi belum memuat rentang. Baris rentang perlu ditambahkan sendiri.


In [ ]:
# Jumlah data per spesies
dataset["species"].value_counts()


In [ ]:
# Ringkasan statistik ditambah baris rentang
summary = dataset.describe()
summary.loc["range"] = summary.loc["max"] - summary.loc["min"]
summary = summary.loc[["mean", "25%", "50%", "75%", "range"]]
summary.rename(index={"50%": "median"})


Ketiga spesies berjumlah sama, masing-masing 50 baris. Nilai rata-rata dan median yang berdekatan menunjukkan sebaran yang cukup simetris pada tiap kolom ukuran.


## Pertanyaan 4

Hitung hal berikut **untuk setiap spesies** pada tabel terpisah:

- Rata-rata setiap ukuran (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`).
- Median setiap ukuran tersebut.

Petunjuk: gunakan `groupby` pada kolom `species` sebelum menghitung statistik. Bila keduanya selesai, coba gabungkan rata-rata dan median dalam satu tabel dengan satu pemanggilan `groupby`.


In [ ]:
# Rata-rata per spesies
dataset.groupby("species").mean(numeric_only=True)


In [ ]:
# Median per spesies
dataset.groupby("species").median(numeric_only=True)


In [ ]:
# Rata-rata dan median dalam satu tabel
dataset.groupby("species").agg(["mean", "median"])


In [ ]:
# Agregasi berbeda untuk kolom tertentu
agg_rules: dict[str, object] = {column: ["mean", "median"] for column in dataset.columns if column != "species"}
agg_rules["petal_length"] = "max"
dataset.groupby("species").agg(agg_rules)


Spesies setosa punya petal paling pendek dan paling lebar, sedangkan virginica punya petal paling panjang. Ukuran sepal ketiga spesies lebih sulit dibedakan dibanding ukuran petal.


## Pertanyaan 5

Buat scatter plot `sepal_length` terhadap `sepal_width` memakai Matplotlib. Beri label pada kedua sumbu dan judul pada plot.


In [ ]:
figure, axis = plt.subplots()
axis.scatter(dataset["sepal_length"], dataset["sepal_width"])
axis.set(
    xlabel="Sepal Length (cm)",
    ylabel="Sepal Width (cm)",
    title="Sepal Length vs Width",
)
plt.show()


## Pertanyaan 6

Buat histogram untuk salah satu dari empat fitur. Beri label sumbu dan judul yang sesuai.


In [ ]:
figure, axis = plt.subplots()
axis.hist(dataset["petal_length"], bins=25)
axis.set(
    xlabel="Petal Length (cm)",
    ylabel="Frequency",
    title="Distribution of Petal Lengths",
)
plt.show()


## Pertanyaan 7

Buat satu plot berisi histogram keempat fitur (`petal_width`, `petal_length`, `sepal_width`, `sepal_length`) yang saling ditumpuk. Setelah itu, coba buat empat histogram terpisah dalam satu gambar.


In [ ]:
# Semua fitur dalam satu bidang plot
sns.set_context("notebook")
axis = dataset.plot.hist(bins=25, alpha=0.5)
axis.set_xlabel("Size (cm)")
plt.show()


In [ ]:
# Empat histogram terpisah dalam satu gambar
axes = dataset.hist(bins=25, figsize=(10, 6))

for row in axes:
    for position, axis in enumerate(row):
        if position == 0:
            axis.set_ylabel("Frequency")
        axis.set_xlabel("Size (cm)")

plt.tight_layout()
plt.show()


Histogram `petal_length` paling jelas terpisah menjadi dua kelompok, tanda bahwa panjang petal sangat membantu membedakan spesies.


## Pertanyaan 8

Dengan pandas, buat boxplot untuk setiap ukuran petal dan sepal.


In [ ]:
dataset.boxplot(by="species", figsize=(10, 6))
plt.tight_layout()
plt.show()


## Pertanyaan 9

Buat satu boxplot dengan fitur pada sumbu x dan spesies dibedakan lewat warna.

Petunjuk: Seaborn membutuhkan data dalam format panjang, yaitu setiap baris berisi satu titik data yang terdiri atas spesies, jenis pengukuran, dan nilainya. Metode `stack` dari pandas bisa menjadi titik awal.


In [ ]:
# Ubah data ke format panjang
plot_data = (
    dataset.set_index("species")
    .stack()
    .to_frame(name="size")
    .reset_index()
    .rename(columns={"level_1": "measurement"})
)
plot_data.head()


In [ ]:
sns.set_style("white")
sns.set_context("notebook")
sns.set_palette("dark")

plt.figure(figsize=(6, 4))
sns.boxplot(x="measurement", y="size", hue="species", data=plot_data)
plt.show()


Boxplot gabungan memperlihatkan bahwa petal lebih tegas memisahkan spesies daripada sepal, dengan setosa berada paling bawah pada kedua ukuran petal.


## Pertanyaan 10

Buat pairplot memakai Seaborn untuk melihat korelasi antar seluruh pengukuran.


In [ ]:
sns.set_context("talk")
sns.pairplot(dataset, hue="species")
plt.show()


Pairplot menegaskan dua pola: `petal_length` dan `petal_width` berkorelasi sangat kuat, sedangkan setosa terpisah jelas dari dua spesies lain hampir pada semua pasangan fitur.
